# Part 3: Bitstream and Deployment with Vitis Unified

In Parts 1 and 2 we converted trained models to hls4ml projects, ran C simulation, and inspected HLS synthesis reports. In this notebook we move one step further: we use the `VitisUnified` backend to wrap the hls4ml design with a board-level AXI interface and produce deployment artifacts for the ZCU102.

This follows the same flow we presented so far, but takes it two steps forward by running implementation, co-simulation, and bitstream generation.

## What Changes in This Part

The normal `Vitis` backend produces an HLS project for the neural network kernel. `VitisUnified` adds the system-facing pieces needed for deployment:

- AXI wrapper around the neural network top function.
- ZCU102 platform/linking workspace.
- Python driver exported for PYNQ-style control.
- The actual hardware link step that produces `system.bit`, `system.hwh`, and implementation reports.

We use the 6-bit QKeras CNN from Part 2 because it is the version that was small enough for the ZCU102 resource budget.

## Setup

This notebook needs an hls4ml build that includes the `VitisUnified` backend. This backend is currently under review in its PR, but our team has used it extensively. The implementation can be found [here](https://github.com/fastmachinelearning/hls4ml/pull/1376).


In [ ]:
from pathlib import Path
import os
import subprocess

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

import hls4ml
import numpy as np
import tensorflow as tf

from qkeras.utils import load_qmodel
from sklearn.metrics import accuracy_score
from tensorflow.keras.datasets import mnist

seed = 0
np.random.seed(seed)
tf.random.set_seed(seed)


def source_settings(settings_path):
    result = subprocess.run(
        ['bash', '-lc', f'source {settings_path} >/dev/null 2>&1 && env'],
        check=True, capture_output=True, text=True,
    )
    os.environ.update(
        line.split('=', 1) for line in result.stdout.splitlines() if '=' in line
    )


vitis_root = os.environ.get('XILINX_VITIS')
settings_candidates = []
if vitis_root is not None:
    settings_candidates.append(Path(vitis_root) / 'settings64.sh')
settings_candidates.extend([
    Path('/opt/Xilinx/Vitis/2024.2/settings64.sh'),
    Path('/opt/Xilinx/Vitis_HLS/2024.2/settings64.sh'),
])

for settings_path in settings_candidates:
    if settings_path.exists():
        source_settings(settings_path)
        print(f'Sourced: {settings_path}')
        break
else:
    print('No Xilinx settings64.sh was found. Emulation can run, but synthesis and bitfile generation need Vitis/Vivado.')


## Load the Test Data and QKeras Model

We only need a small evaluation slice here. The goal of Part 3 is the hardware build/deployment flow, not retraining or measuring final test-set accuracy again.

In [ ]:
(_, _), (X_test, y_test) = mnist.load_data()

X_eval = X_test[:1000].astype('float32') / 255.0
X_eval = np.ascontiguousarray(X_eval.reshape(-1, 28, 28, 1))
y_eval = y_test[:1000]

model_path = Path('models/mnist_cnn_qkeras_6bit.keras')
if not model_path.exists():
    raise FileNotFoundError(f'{model_path} not found. Run Part 2 first or provide the prepared QKeras model.')

model = load_qmodel((model_path))
print(f'Loaded QKeras model from {model_path}')

model.summary()
print('Model name:', model.name)
print('Evaluation batch:', X_eval.shape)

In [ ]:
demo_inputs = X_eval
models_dir = Path('models')
name = model_path.stem

y_keras = model.predict(demo_inputs, verbose=0)
y_keras_labels = np.argmax(y_keras, axis=1)
print('QKeras accuracy on evaluation slice: {:.4f}'.format(accuracy_score(y_eval, y_keras_labels)))

np.save(models_dir / f'X_test_{name}.npy', demo_inputs)
np.save(models_dir / f'labels_test_{name}.npy', y_test[: len(demo_inputs)])
np.save(models_dir / f'y_test_{name}.npy', y_keras)

print('Saved evaluation arrays:')
print(models_dir / f'X_test_{name}.npy')
print(models_dir / f'labels_test_{name}.npy')
print(models_dir / f'y_test_{name}.npy')

## Generate the hls4ml Configuration

In [ ]:
hls_config = hls4ml.utils.config_from_keras_model(model, granularity='name')

PROJECT_NAME = 'mnist_q6_project'
OUTPUT_DIR = Path(f'hls4ml_project_{model_path.stem}_unified').resolve()
XPFM_PATH = Path(os.environ.get('XILINX_VITIS', '/opt/Xilinx/Vitis/2023.2')) / 'base_platforms/xilinx_zcu102_base_202320_1/xilinx_zcu102_base_202320_1.xpfm'

## Convert with `VitisUnified`

The conversion writes the normal hls4ml firmware plus extra files for the board-level Vitis flow. After this cell, the project directory should already contain the AXI wrapper, the linker script, and the generated Python driver.

The following settings are applied:

- `backend='VitisUnified'` selects the full Vitis Unified flow.
- `board='zcu102'` selects the ZCU102 platform templates.
- `axi_mode='axi_stream'` builds a streaming wrapper with DMA.
- `input_type='float'` and `output_type='float'` keep the PS-to-PL interface simple for the tutorial. The internal neural-network arithmetic still follows the hls4ml/QKeras precisions.

In [ ]:
hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=hls_config,
    output_dir=str(OUTPUT_DIR),
    project_name=PROJECT_NAME,
    backend='VitisUnified',
    board='zcu102',
    part='xczu9eg-ffvb1156-2-e',
    clock_period='5ns',
    io_type='io_stream',
    input_type='float',
    output_type='float',
    xpfmPath=str(XPFM_PATH),
    axi_mode='axi_stream',
)

hls_model.compile()
print('Generated and compiled:', OUTPUT_DIR)

## Run the Vitis Unified Build

This is the expensive step. It runs co-simulation, synthesis, packaging, and the full hardware link.

- `csim` runs C simulation of the wrapper.
- `cosim` runs RTL co-simulation.
- `synth` compiles/packages the HLS kernel as a Vitis object.
- `bitfile` runs the Vitis link step, creates the block design, and exports `system.bit` plus `system.hwh`.

The script-style flow is still one `hls_model.build(...)` call.

In [ ]:
hls_model.build(
    csim=True,
    cosim=True,
    synth=True,
    bitfile=True,
)

## Build Artifacts

After the full build, the deployment-critical files are:

- `export/system.bit`: FPGA bitstream extracted from the linked `.xclbin`.
- `export/system.hwh`: hardware handoff metadata used by PYNQ.
- `export/axi_stream_driver.py`: generated Python driver for the AXI-stream accelerator.
- `final_reports/*.rpt`: implementation resource and timing reports.

## Prepare a Deployment Folder

The files that we need in order to exeacute the design on a board with a PYNQ image are: the bitstream, hardware handoff, generated driver, and a evaluation batch, that we generated during the inference step of this tutorial.

The following files can be transfered on board using `scp` or by manually creating a folder and uploading them through the GUI:
- export/system.bit
- export/system.hwh
- export/axi_stream_driver.py
- models/X_test_mnist_cnn_qkeras_6bit.npy
- models/y_test_mnist_cnn_qkeras_6bit.npy
- models/labels_test_mnist_cnn_qkeras_6bit.npy

## Board-Side Validation

The following code is intended to run on the ZCU102/PYNQ environment after copying the deployment package. Keep it as a reference block in this build notebook so the host-side notebook can still execute without requiring `pynq`.

```python
from axi_stream_driver import NeuralNetworkOverlay
import numpy as np
X_eval = np.load('X_eval.npy')
y_eval = np.load('y_eval.npy')

nn = NeuralNetworkOverlay('system.bit', X_eval.shape, (X_eval.shape[0], 10))
y_hw, latency, throughput = nn.predict(X_eval, profile=True)

print('Hardware accuracy:', np.mean(np.argmax(y_hw, axis=1) == y_eval))
print('Latency:', latency)
print('Throughput:', throughput)
```
